# Cleaning – Data Loading & Preprocessing

Pipeline này thực hiện đọc dữ liệu từ 2 kỳ (April–June và July–September), hợp nhất, align cột, làm sạch, kiểm tra validation, tạo đặc trưng mới, xử lý outlier, impute missing values, encode categorical và scale numeric.

| Bước | Kỹ thuật chính |
|------|----------------|
| Data Loading | Merge 4 CSVs, gán nhãn delay/not-delay, align cột giữa 2 kỳ |
| Validation | Blank spaces, giá trị âm, cột low-variance |
| Feature Engineering | Time-based features (time_period, IS_WEEKEND...), binning (Order_Size_Group, Value_Group) |
| Outlier Handling | Winsorization IQR-based clipping |
| Preprocessing | Drop duplicates/ID columns, impute median/"__MISSING__", LabelEncoder, StandardScaler |


## 1.1 Config

Định nghĩa các đường dẫn dữ liệu, tên cột datetime, cột cần loại bỏ, và seed ngẫu nhiên để đảm bảo reproducibility giữa các lần chạy.


In [ ]:
"""Configuration constants for DS108 Lab 4 pipeline."""
import os

# Paths
DATA_DIR = "Data"
RESULTS_DIR = "results"
EDA_DIR = os.path.join(RESULTS_DIR, "eda")
EXP_DIR = os.path.join(RESULTS_DIR, "experiments")
MODEL_DIR = os.path.join(RESULTS_DIR, "models")
PLOT_DIR = os.path.join(RESULTS_DIR, "plots")

# Raw files
DELAY_46 = os.path.join(DATA_DIR, "delay_4_6_CONDITION_PRODUCT_SUPPLIER.csv")
NOT_DELAY_46 = os.path.join(DATA_DIR, "not_delay_4_6_CONDITION_PRODUCT_SUPPLIER.csv")
DELAY_79 = os.path.join(DATA_DIR, "delay_7_9_CONDITION_PRODUCT_SUPPLIER.csv")
NOT_DELAY_79 = os.path.join(DATA_DIR, "not_delay_7_9_CONDITION_PRODUCT_SUPPLIER.csv")

# Experiment labels
PERIOD_A = "7_9"   # future
PERIOD_B = "4_6"   # past

# Random seed for reproducibility
RANDOM_STATE = 42

# Train/Val/Test split ratios
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

# Stratified K-Fold
N_SPLITS = 5

# Incremental learning ratios for alpha_4
INCREMENTAL_RATIOS = [0.1, 0.3, 0.5, 0.7, 0.9]

# Model light hyperparameters (tuned via CV inside training)
LGBM_PARAMS = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "num_leaves": 31,
    "learning_rate": 0.05,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
    "n_estimators": 1000,
    "random_state": RANDOM_STATE,
    "is_unbalance": True,
}

XGB_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.9,
    "n_estimators": 1000,
    "random_state": RANDOM_STATE,
}

CAT_PARAMS = {
    "iterations": 1000,
    "learning_rate": 0.05,
    "depth": 6,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "random_seed": RANDOM_STATE,
    "verbose": False,
    "auto_class_weights": "Balanced",
}

KNN_PARAMS = {
    "n_neighbors": 5,
}

## 1.2 Data Loader

Đọc 4 file CSV (delay + not-delay cho 2 kỳ), gán nhãn `DELAY_FLG=1` cho delay và `DELAY_FLG=0` cho not-delay, sau đó align cột giữa 2 kỳ.

- Kỳ 4–6 có **47 cột**, kỳ 7–9 có **37 cột** → intersection = **37 cột chung**.
- 10 cột dư ở kỳ 4–6 (ví dụ: `ACTUAL_SHIP_DAYS`, `HEAVY_FLG`, `HAZARD_FLG`, `SUPPLIER_CATEGORY_CD`...) bị loại bỏ để đảm bảo consistency giữa 2 kỳ.
- Tổng số mẫu sau merge: ~2.9M rows (4–6) + ~6.3M rows (7–9).


In [ ]:
"""Data loading and initial merging for both time periods."""
import pandas as pd
import config

def load_period(delay_path: str, not_delay_path: str, label: int) -> pd.DataFrame:
    """Load delay and not_delay files for a period and concatenate."""
    df_delay = pd.read_csv(delay_path, low_memory=False)
    df_not = pd.read_csv(not_delay_path, low_memory=False)
    df_delay["label"] = label
    df_not["label"] = 1 - label
    df = pd.concat([df_delay, df_not], ignore_index=True)
    return df

def load_all() -> dict:
    """Load and return both periods as dict {period_name: DataFrame}."""
    df_46 = load_period(config.DELAY_46, config.NOT_DELAY_46, label=1)
    df_79 = load_period(config.DELAY_79, config.NOT_DELAY_79, label=1)
    print(f"Loaded period 4-6: {df_46.shape}")
    print(f"Loaded period 7-9: {df_79.shape}")
    return {"4_6": df_46, "7_9": df_79}

def align_columns(df_dict: dict) -> dict:
    """Align columns across periods using only common columns.

    4-6 has 10 extra columns not present in 7-9. For cross-period
    experiments we must use the intersection to avoid leakage/mismatch.
    """
    cols_46 = set(df_dict["4_6"].columns)
    cols_79 = set(df_dict["7_9"].columns)
    common = list(cols_46 & cols_79)
    common.sort()
    print(f"Common columns: {len(common)}")
    print(f"Dropped from 4-6: {cols_46 - cols_79}")
    aligned = {}
    for period, df in df_dict.items():
        aligned[period] = df[common].copy()
        print(f"Aligned {period}: {aligned[period].shape}")
    return aligned

def add_period_flag(df_dict: dict) -> dict:
    """Add a metadata column indicating the source period."""
    for period, df in df_dict.items():
        df["__period"] = period
    return df_dict

def build_datasets() -> dict:
    """Full pipeline: load, align, flag."""
    df_dict = load_all()
    df_dict = align_columns(df_dict)
    df_dict = add_period_flag(df_dict)
    return df_dict

datasets = build_datasets()
for name, df in datasets.items():
    print(f"\n{name}: {df.shape}")
    print(df["label"].value_counts())

### Lưu ý về cấu trúc dữ liệu

Dữ liệu được chia thành 2 kỳ để đánh giá khả năng **generalize** của mô hình qua thời gian. Việc align cột là bắt buộc vì nếu train trên tập có feature mà test không có, model sẽ crash hoặc ignore feature đó một cách không kiểm soát.

Cột `SPECIAL DIV` (dấu cách) và `SPECIAL_DIV` (gạch dưới) trong CSV gốc tạo ra collision. Sau khi sanitize, cột thứ hai được đổi tên thành `SPECIAL_DIV_2`.

## 1.3 Data Validation

Trước khi làm sạch, ta kiểm tra các vấn đề tiềm ẩn trong dữ liệu thô:

1. **Blank spaces** trong cột string: có thể gây lỗi khi groupby hoặc encode.
2. **Giá trị âm** trong cột numeric: một số cột như `SO QTY` không nên có giá trị âm.
3. **Cột low-variance** (>99% giá trị giống nhau): không mang thông tin phân biệt, loại bỏ để giảm chiều dữ liệu và tránh overfitting.

> Tương tự notebook tham khảo, ta cũng xử lý cột `OTHER AREA SHIP DIV` có giá trị whitespace là hợp lệ (không phải lỗi).


In [ ]:
"""Preprocessing pipeline: cleaning, encoding, scaling, feature engineering."""
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import config

DROP_COLS = [
    "Order_date", "VSD", "GLOBAL_NO", "CUST_CD", "PRODUCT_CD",
    "SUPPLIER_CD", "SHIP_DECISION_NO", "SOUF_RCV_NO", "QTUF_RCV_NO",
    "REASON_CD", "ACTUAL_SHIP_DAYS", "PRODUCT_ASSORT", "__period",
]

CATEGORICAL_COLS = [
    "SUBSIDIARY_CD", "CLASSIFY_CD", "BRAND_CD", "INNER_CD",
    "Stock_class", "Consider_count_hodiday_Saturday",
    "OTHER_AREA_SHIP_DIV", "PACKING_RANK", "PRODUCT_ATTRIBUTION",
    "SPECIAL_DIV", "SPECIAL_DIV_2", "LOGICAL_PLANT", "DIRECT_SHIP_FLG",
    "DELI_DIV", "Ship_Mode", "SUPPLIER_DIV",
    "SO_DAY_OF_MONTH", "SO_DAY_OF_WEEK", "SO_TIME",
]

def sanitize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    new_cols = []
    seen = {}
    for c in df.columns:
        clean = c.replace(" ", "_").replace("-", "_")
        if clean in seen:
            seen[clean] += 1
            clean = f"{clean}_{seen[clean]}"
        else:
            seen[clean] = 1
        new_cols.append(clean)
    df.columns = new_cols
    return df

def detect_blank_spaces(df: pd.DataFrame) -> dict:
    str_cols = df.select_dtypes(include="object").columns
    ratios = {}
    for col in str_cols:
        ratio = df[col].dropna().apply(
            lambda x: isinstance(x, str) and x.strip() == ""
        ).mean()
        if ratio > 0:
            ratios[col] = ratio
    return ratios

def validate_negative_values(df: pd.DataFrame, num_cols: list = None) -> dict:
    if num_cols is None:
        num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    invalid = {}
    for col in num_cols:
        if col == "label":
            continue
        neg_count = (df[col] < 0).sum()
        if neg_count > 0:
            invalid[col] = int(neg_count)
    return invalid

def drop_low_variance_columns(df: pd.DataFrame, threshold: float = 0.99) -> pd.DataFrame:
    df = df.copy()
    dropped = []
    for col in df.columns:
        top_freq = df[col].value_counts(normalize=True, dropna=False).values[0]
        if top_freq >= threshold:
            dropped.append(col)
    if dropped:
        df = df.drop(columns=dropped)
        print(f"Dropped low-variance columns: {dropped}")
    return df

# Demo on raw 4-6 data
df_demo = datasets["4_6"].copy()
df_demo = sanitize_columns(df_demo)

print("=== Blank space detection ===")
blanks = detect_blank_spaces(df_demo)
for col, ratio in sorted(blanks.items(), key=lambda x: x[1], reverse=True):
    print(f"  {col}: {ratio:.2%} cells are blank spaces only")

print("\n=== Negative value validation ===")
negs = validate_negative_values(df_demo)
print(negs if negs else "No negative values found.")

print("\n=== Low variance columns ===")
df_demo = drop_low_variance_columns(df_demo, threshold=0.99)

### Kết quả validation thường gặp

- Cột `SPECIAL DIV` (space) và `SPECIAL_DIV` (underscore) bị trùng tên sau sanitize → được đổi tên thành `SPECIAL_DIV_2`.
- Cột `DIRECT_SHIP_FLG` và `SPECIAL_DIV` có độ biến thiên rất thấp → cân nhắc loại bỏ.
- Một số cột numeric có outliers rõ rệt (right-skewed) → cần Winsorization ở bước sau.


## 1.4 Feature Engineering

Tạo các đặc trưng mới từ dữ liệu thời gian và phân nhóm giá trị numeric:

**Time-based features:**
- `time_period`: Morning/Afternoon/Evening/Night từ `SO_TIME`
- `IS_WEEKEND`: binary (0/1) từ `SO_DAY_OF_WEEK`
- `MONTH_PHASE`: early/mid/late từ `SO_DAY_OF_MONTH`
- `Order_month`, `VSD_month`: tháng từ Order date và VSD
- `VSD_IS_WEEKEND`, `VSD_MONTH_PHASE`: weekend/phase từ VSD
- `Expected_delivery_days`: số ngày giao hàng dự kiến (`VSD` − Order date)

**Binning features:**
- `Order_Size_Group`: Small/Medium/Large từ `SO QTY` (pd.qcut)
- `Value_Group`: Low/Medium/High từ `SUPPLIER INV AMOUNT` (pd.qcut)

> `SO_TIME` ở định dạng HHMMSS (ví dụ: 120324 = 12:03:24). Để parse đúng, ta chuyển về string zero-pad 6 chữ số rồi extract giờ.


In [ ]:
def create_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in ["Order date", "Order_date", "VSD"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    if "SO_TIME" in df.columns:
        df["SO_TIME_str"] = df["SO_TIME"].astype(str).str.zfill(6)
        hour = pd.to_datetime(df["SO_TIME_str"], format="%H%M%S", errors="coerce").dt.hour
        df["time_period"] = hour.apply(
            lambda h: "Morning" if 5 <= h < 12 else
                      "Afternoon" if 12 <= h < 17 else
                      "Evening" if 17 <= h < 22 else
                      "Night" if pd.notna(h) else "__MISSING__"
        )

    if "SO_DAY_OF_WEEK" in df.columns:
        df["IS_WEEKEND"] = df["SO_DAY_OF_WEEK"].isin([5, 6]).astype(int)

    if "SO_DAY_OF_MONTH" in df.columns:
        df["MONTH_PHASE"] = pd.cut(
            df["SO_DAY_OF_MONTH"], bins=[0, 7, 14, 31],
            labels=["early", "mid", "late"]
        ).astype(str)

    order_date_col = None
    for c in ["Order date", "Order_date"]:
        if c in df.columns:
            order_date_col = c
            break
    if order_date_col is not None:
        df["Order_month"] = df[order_date_col].dt.month

    if "VSD" in df.columns:
        df["VSD_month"] = df["VSD"].dt.month
        df["VSD_IS_WEEKEND"] = df["VSD"].dt.dayofweek.isin([5, 6]).astype(int)
        df["VSD_MONTH_PHASE"] = pd.cut(
            df["VSD"].dt.day, bins=[0, 7, 14, 31],
            labels=["early", "mid", "late"]
        ).astype(str)

    if order_date_col is not None and "VSD" in df.columns:
        df["Expected_delivery_days"] = (df["VSD"] - df[order_date_col]).dt.days

    return df

def create_binning_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    qty_col = "SO QTY" if "SO QTY" in df.columns else "SO_QTY"
    if qty_col in df.columns:
        df["Order_Size_Group"] = pd.qcut(
            df[qty_col], q=3, duplicates="drop", labels=["Small", "Medium", "Large"]
        ).astype(str)

    inv_col = None
    for c in ["SUPPLIER INV AMOUNT", "SUPPLIER_INV_AMOUNT"]:
        if c in df.columns:
            inv_col = c
            break
    if inv_col:
        df["Value_Group"] = pd.qcut(
            df[inv_col], q=3, duplicates="drop", labels=["Low", "Medium", "High"]
        ).astype(str)
    return df

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = create_time_features(df)
    df = create_binning_features(df)
    return df

# Demo
df_fe = engineer_features(datasets["4_6"].copy())
new_cols = [c for c in df_fe.columns if c not in datasets["4_6"].columns]
print("New engineered columns:", new_cols)
print(df_fe[new_cols].head())

### Tại sao cần Feature Engineering?

Các mô hình tree-based (LGBM, XGB, CatBoost) có thể tự học pattern từ timestamp thô, nhưng việc tạo feature rõ ràng như `time_period` hay `IS_WEEKEND` giúp:
- Giảm độ phức tạp cần thiết của cây quyết định
- Tăng tính interpretability (có thể giải thích tại sao weekend lại ảnh hưởng đến delay)
- Cải thiện khả năng generalize khi timestamp nằm ngoài range train


## 1.5 Outlier Handling – Winsorization

Sử dụng **IQR method** để clip outliers thay vì xóa, giữ nguyên số lượng mẫu.

Công thức:
- `lower = Q1 − 1.5 × IQR`
- `upper = Q3 + 1.5 × IQR`

> Lưu ý: Bỏ qua các cột có ≤10 giá trị unique (tránh clip binary/categorical như `DIRECT_SHIP_FLG`, `SPECIAL_DIV`).


In [ ]:
def winsorize_column(df: pd.DataFrame, column: str,
                     lower_q: float = 0.25, upper_q: float = 0.75,
                     multiplier: float = 1.5) -> pd.DataFrame:
    df = df.copy()
    Q1 = df[column].quantile(lower_q)
    Q3 = df[column].quantile(upper_q)
    IQR = Q3 - Q1
    lower = Q1 - multiplier * IQR
    upper = Q3 + multiplier * IQR
    original_outliers = ((df[column] < lower) | (df[column] > upper)).sum()
    df[column] = df[column].clip(lower, upper)
    if original_outliers > 0:
        print(f"[{column}] Winsorized: {original_outliers} outliers clipped to [{lower:.2f}, {upper:.2f}]")
    return df

def winsorize_df(df: pd.DataFrame, columns: list = None,
                 lower_q: float = 0.25, upper_q: float = 0.75,
                 multiplier: float = 1.5) -> pd.DataFrame:
    df = df.copy()
    if columns is None:
        columns = df.select_dtypes(include=[np.number]).columns.tolist()
        if "label" in columns:
            columns.remove("label")
    for col in columns:
        if col in df.columns and pd.api.types.is_numeric_dtype(df[col]):
            df = winsorize_column(df, col, lower_q, upper_q, multiplier)
    return df

# Demo on numeric columns of 4-6 (explicit columns, auto-skips categorical-like)
df_winsor = datasets["4_6"].copy()
demo_cols = ["SO QTY", "SUPPLIER INV AMOUNT", "WEIGHT PER PIECE",
             "PURCHASE AMOUNT", "Sales order line number"]
df_winsor = winsorize_df(df_winsor, columns=demo_cols)

### Tại sao dùng Winsorization thay vì xóa outliers?

- **Giữ nguyên kích thước dataset**: quan trọng với dữ liệu imbalance, mỗi mẫu minority đều quý.
- **Không thay đổi phân phối quá mức**: chỉ clip phần đuôi, không biến đổi toàn bộ như log-transform.
- **Tree-based models vẫn hưởng lợi**: dù robust với outliers, việc giảm range giúp split tốt hơn.


## 1.6 Full Preprocessing Pipeline

Pipeline hoàn chỉnh thực hiện tuần tự các bước:

1. **Drop ID columns**: `GLOBAL NO`, `PRODUCT_CD`, `SHIP DECISION NO` — có tính unique cao, dễ gây overfitting do model memorize thay vì generalize.
2. **Drop duplicates**: loại bỏ rows trùng lặp hoàn toàn.
3. **Handle missing**: median cho numeric, `"__MISSING__"` cho categorical.
4. **Encode**: LabelEncoder với unseen category mapping về −1.
5. **Scale**: StandardScaler fit trên train only, transform cả 3 tập.
6. **Split**: Stratified 8:1:1 (train/val/test) để giữ tỷ lệ lớp ổn định.


In [ ]:
def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = sanitize_columns(df)
    # Clean specific column
    col = "OTHER_AREA_SHIP_DIV"
    if col in df.columns:
        df[col] = df[col].astype(str).str.replace(".0", "", regex=False)
        df[col] = df[col].replace(["nan", "NaN", " "], "0")
        df[col] = df[col].replace({"1.0": "1", "1": "1"})
    drop_existing = [c for c in DROP_COLS if c in df.columns]
    df = df.drop(columns=drop_existing)
    before = len(df)
    df = df.drop_duplicates()
    if len(df) < before:
        print(f"Dropped {before - len(df)} duplicate rows")
    return df

def handle_missing(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in df.columns:
        if col == "label":
            continue
        if df[col].dtype == object:
            df[col] = df[col].where(pd.notna(df[col]), "__MISSING__")
        else:
            df[col] = df[col].fillna(df[col].median())
    return df

def reduce_memory(df: pd.DataFrame) -> pd.DataFrame:
    for col in df.columns:
        if col == "label" or df[col].dtype == object:
            continue
        if pd.api.types.is_integer_dtype(df[col]):
            df[col] = pd.to_numeric(df[col], downcast="integer")
        else:
            df[col] = pd.to_numeric(df[col], downcast="float")
    return df

def encode_categoricals(df: pd.DataFrame, encoders: dict = None, fit: bool = True) -> tuple:
    df = df.copy()
    if encoders is None:
        encoders = {}
    for col in CATEGORICAL_COLS:
        if col not in df.columns:
            continue
        if fit:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            encoders[col] = le
        else:
            le = encoders.get(col)
            if le:
                mapping = {cls: idx for idx, cls in enumerate(le.classes_)}
                df[col] = df[col].astype(str).map(mapping).fillna(-1).astype(int)
    # Encode any remaining object columns
    for col in df.columns:
        if col == "label":
            continue
        if df[col].dtype == object and col not in encoders:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            encoders[col] = le
    return df, encoders

def scale_numeric(df: pd.DataFrame, scaler: StandardScaler = None, fit: bool = True) -> tuple:
    df = df.copy()
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if "label" in numeric_cols:
        numeric_cols.remove("label")
    if fit:
        scaler = StandardScaler()
        df[numeric_cols] = scaler.fit_transform(df[numeric_cols])
    else:
        df[numeric_cols] = scaler.transform(df[numeric_cols])
    return df, scaler

def split_data(df: pd.DataFrame, val_ratio: float = config.VAL_RATIO,
               test_ratio: float = config.TEST_RATIO, random_state: int = config.RANDOM_STATE):
    y = df["label"]
    X = df.drop(columns=["label"])
    X_trainval, X_test, y_trainval, y_test = train_test_split(
        X, y, test_size=test_ratio, stratify=y, random_state=random_state
    )
    if val_ratio > 0:
        val_ratio_adjusted = val_ratio / (1 - test_ratio)
        X_train, X_val, y_train, y_val = train_test_split(
            X_trainval, y_trainval, test_size=val_ratio_adjusted,
            stratify=y_trainval, random_state=random_state
        )
    else:
        X_train, y_train = X_trainval, y_trainval
        X_val, y_val = None, None
    return (X_train, y_train), (X_val, y_val), (X_test, y_test)

def preprocess_pipeline(df: pd.DataFrame, encoders=None, scaler=None, fit=True,
                        do_feature_engineering: bool = True,
                        do_winsorize: bool = False) -> tuple:
    if do_feature_engineering:
        df = engineer_features(df)
    df = clean_dataframe(df)
    df = handle_missing(df)
    if do_winsorize:
        df = winsorize_df(df)
    df = reduce_memory(df)
    df, encoders = encode_categoricals(df, encoders=encoders, fit=fit)
    df, scaler = scale_numeric(df, scaler=scaler, fit=fit)
    return df, encoders, scaler

# Run full pipeline on both periods
for period, df in datasets.items():
    df_proc, enc, scl = preprocess_pipeline(df, fit=True)
    print(f"{period} processed shape: {df_proc.shape}")
    print(f"  Columns: {list(df_proc.columns)}")

---

## Export

Dữ liệu sau preprocessing được lưu lại để sử dụng cho EDA và Modeling ở các bước sau.
Các artifacts như `missing_values.csv` và `numeric_summary.csv` cũng được xuất ra thư mục `results/eda/`.
